# HRV Feature Extraction (Multi Subjects)
---
## Goal
In the previous notebook `03_ecg_rr_pipeline.ipynb`, I completed the ECG → R-peak → RR interval pipeline and generated subject level RR files like `rr_S2.csv`.

This notebook is going to perform the **physiological encoding stage of Echo**. The objective is to transform raw RR intervals into structured **heart rate variability (HRV) feature vectors**, computed within sliding windows that have strict length. Each window represents a fundamental unit that will later be mapped into symbolic emotion tokens.

### In this notebook I'm going to

1. Load RR interval files for multiple subjects  
2. Define a consistent sliding window scheme  
3. Compute time domain and frequency domain HRV features  
4. Perform basic window level quality control  
5. Aggregate features across subjects  
6. Save the final output as `features_hrv.csv`

---
## 1. Set up

This section imports all required libraries and defines the file paths used throughout the notebook. RR interval files are generated in the previous stage and stored in the results directory.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import neurokit2 as nk

data_path = Path("../results")    # this is the path where RR data is stored      
output_path = Path("../results")  # this is the path where features will be stored
output_path.mkdir(exist_ok=True)  # this is to make the output directory 
print("RR data path:", data_path)
print("Output path:", output_path)

RR data path: ../results
Output path: ../results


---
## 2. Build Helper Function

In this section, I define helper functions for HRV computation. These functions operate on RR interval sequences and return quantitative descriptors of autonomic nervous system activity. The extracted features will serve as the physiological basis of Echo’s emotion language system.

---

### 2.1 Time domain HRV features

Time domain HRV features look at how heartbeats change from one beat to the next. They provide a straightforward way to understand cardiac variability and stability.

In this step, several commonly used HRV features are extracted.

- **Mean NN interval** ： the average time between heartbeats  
- **SDNN** ： overall variability across the window  
- **RMSSD** ： short-term beat-to-beat variability  
- **pNN50** ： percentage of large changes between consecutive beats

In [6]:
def compute_time_domain_hrv(rr_sec):
    """
    Compute basic time-domain HRV features.

    Parameters
    ----------
    rr_sec : array-like
        RR intervals in seconds

    Returns
    -------
    mean_nn, sdnn, rmssd, pnn50
    """
    rr = np.array(rr_sec)    # this is to ensure rr is a numpy array that we can work with

    # this is to handle cases where rr has less than 2 values
    if len(rr) < 2:
        return np.nan, np.nan, np.nan, np.nan    # here we are returning nan for all features if there are not enough rr intervals

    mean_nn = rr.mean()
    sdnn = rr.std(ddof=1)  # we are using sample standard deviation to be consistent with neurokit2

    diff_rr = np.diff(rr)   # the differences between successive rr intervals are needed for rmssd and pnn50
    rmssd = np.sqrt(np.mean(diff_rr ** 2)) if len(diff_rr) > 0 else np.nan     # rmssd is the root mean square of successive differences
    pnn50 = np.mean(np.abs(diff_rr) > 0.05)      # pnn50 is the proportion of successive differences greater than 0.05 seconds

    return mean_nn, sdnn, rmssd, pnn50


---

### 2.2 Frequency domain HRV features

Frequency domain HRV features describe how heart rate variability is spread across different frequency bands. These features are often used as an indicator of autonomic balance. Here, the LF/HF ratio is calculated using Welch spectral estimation provided by NeuroKit.

In [7]:
def compute_lf_hf(rr_sec):
    """
    Compute LF/HF ratio using NeuroKit.

    RR input must be in milliseconds.
    """
    if len(rr_sec) < 4:
        return np.nan

    rr_ms = np.array(rr_sec) * 1000    # this is to convert rr intervals to milliseconds, because neurokit2 expects rr intervals in ms

    # the purpose here is to catch any error that may arise during the computation
    try:
        hrv_freq = nk.hrv_frequency(rr_ms, show=False)
        return hrv_freq.get("HRV_LFHF", np.nan)
    except Exception:
        return np.nan


---

### 2.3 Window level quality metrics

In this step, a basic quality score is computed for each window based on whether the RR intervals fall within a reasonable physiological range. This helps flag unstable segments while keeping the full signal available
for later analysis.

In [8]:
def compute_window_quality(rr_sec, rr_min=0.3, rr_max=2.0):
    """
    Simple RR quality scoring.
    """
    rr = np.array(rr_sec)

    if len(rr) == 0:   # this case needs to be handled to avoid division by zero
        return np.nan

    outliers = (rr < rr_min) | (rr > rr_max)  # outliers are rr intervals that are too short or too long
    p_outlier = outliers.mean()

    quality_score = 1 - p_outlier    # quality score is 1 minus the proportion of outliers according to the definition
    quality_score = np.clip(quality_score, 0, 1)      # this is to ensure quality score is between 0 and 1

    return quality_score


---
## 3. Sliding window extraction (per subject)

Here, RR intervals are segmented into overlapping time windows. Each window represents a short snapshot of physiological activity, allowing changes over time to be examined more clearly.

This window based representation makes it easier to track emotional dynamics
as a sequence of meaningful states rather than a continuous raw signal.

---

### Sliding window configuration

- Window length: **60 seconds**  
- Step size: **30 seconds**

This configuration offers a practical tradeoff between resolution and
physiological reliability.

In [5]:
WINDOW_SIZE = 60
STEP_SIZE = 30
MIN_BEATS = 5

In [9]:
def extract_hrv_windows(subj):
    """
    Extract HRV features from one subject.

    Returns one row per sliding window.
    """
    rr_file = data_path / f"rr_{subj}.csv"

    if not rr_file.exists():
        print(f"[Missing] {rr_file}")
        return pd.DataFrame()

    df_rr = pd.read_csv(rr_file)

    # this is to ensure the required columns are present
    t_min = df_rr["time_sec"].min()
    t_max = df_rr["time_sec"].max()

    windows = []     # this will hold the results for all windows
    window_id = 0
    t_start = t_min

    while t_start + WINDOW_SIZE <= t_max:
        t_end = t_start + WINDOW_SIZE

        mask = (df_rr["time_sec"] >= t_start) & (df_rr["time_sec"] < t_end)
        rr_win = df_rr.loc[mask, "RR_sec"].values
        hr_win = df_rr.loc[mask, "HR_bpm"].values

        if len(rr_win) >= MIN_BEATS:
            mean_nn, sdnn, rmssd, pnn50 = compute_time_domain_hrv(rr_win)
            lf_hf = compute_lf_hf(rr_win)
            hr_mean = hr_win.mean() if len(hr_win) > 0 else np.nan
            quality = compute_window_quality(rr_win)

            windows.append({
                "subject": subj,
                "window_id": window_id,
                "t_start": t_start,
                "t_end": t_end,
                "MeanNN": mean_nn,
                "SDNN": sdnn,
                "RMSSD": rmssd,
                "pNN50": pnn50,
                "HR_mean": hr_mean,
                "LF_HF": lf_hf,
                "quality_score": quality,
                "n_beats": len(rr_win)
            })

            window_id += 1

        t_start += STEP_SIZE

    return pd.DataFrame(windows)


---
## 4. Run multi subject extraction

Here, the HRV extraction pipeline is run across all selected WESAD subjects.

Each subject is processed separately, and the extracted windows are later
merged into one consolidated feature table.

This creates a consistent physiological representation that can be carried
forward into the lexicon layer.

In [13]:
subjects = [
    "S2","S3","S4","S5","S6","S7","S8",
    "S9","S10","S11","S13","S14","S15"
]

all_subjects = []

for subj in subjects:
    print(f"Processing {subj}...")
    df_subj = extract_hrv_windows(subj)

    if not df_subj.empty:
        all_subjects.append(df_subj)

Processing S2...
Processing S3...
[Missing] ../results/rr_S3.csv
Processing S4...
[Missing] ../results/rr_S4.csv
Processing S5...
[Missing] ../results/rr_S5.csv
Processing S6...
[Missing] ../results/rr_S6.csv
Processing S7...
[Missing] ../results/rr_S7.csv
Processing S8...
[Missing] ../results/rr_S8.csv
Processing S9...
[Missing] ../results/rr_S9.csv
Processing S10...
[Missing] ../results/rr_S10.csv
Processing S11...
[Missing] ../results/rr_S11.csv
Processing S13...
[Missing] ../results/rr_S13.csv
Processing S14...
[Missing] ../results/rr_S14.csv
Processing S15...
[Missing] ../results/rr_S15.csv


In [14]:
features_hrv = pd.concat(all_subjects, ignore_index=True)
features_hrv.head()


,subject,window_id,t_start,t_end,MeanNN,SDNN,RMSSD,pNN50,HR_mean,LF_HF,quality_score,n_beats
0,S2,0,1.725714,61.725714,0.790207,0.085074,0.043753,0.213333,76.791505,NaN,1.0,76
1,S2,1,31.725714,91.725714,0.759747,0.060547,0.040032,0.166667,79.465489,NaN,1.0,79
2,S2,2,61.725714,121.725714,0.790376,0.051677,0.040975,0.200000,76.237465,NaN,1.0,76
3,S2,3,91.725714,151.725714,0.815077,0.053393,0.044399,0.315068,73.921045,NaN,1.0,74
4,S2,4,121.725714,181.725714,0.807600,0.066821,0.049256,0.337838,74.806137,NaN,1.0,75


---
## 5. Save features_hrv.csv
The final output of this notebook is a consolidated HRV feature table.

In [15]:
output_file = output_path / "features_hrv.csv"
features_hrv.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Total windows:", len(features_hrv))


Saved: ../results/features_hrv.csv
Total windows: 201
